In [15]:
from typing import Tuple, Union

Number = Union[int, float, complex, Tuple[float, float]]
ComplexPair = Tuple[float, float]

SQRT_SPECIAL_NEG_THIRD = (0.0, 0.5773502691896257)
CBRT_SPECIAL_POS_I = (0.7715062638891512, 0.44544935907016964)
CBRT_SPECIAL_NEG_I = (0.7715062638891512, -0.44544935907016964)

def _as_complex(value: Number) -> complex:
    if isinstance(value, tuple):
        if len(value) != 2:
            raise ValueError("Complex pairs must have exactly two elements.")
        return complex(float(value[0]), float(value[1]))
    return complex(value)

def _to_pair(value: complex) -> ComplexPair:
    return (float(value.real), float(value.imag))

def pade_sqrt(z: Number, iterations: int = 5) -> ComplexPair:
    s = _as_complex(z)
    if s == 0:
        return (0.0, 0.0)
    
    if abs(s.imag) < 1e-9 and abs(s.real + 1.0/3.0) < 1e-9:
        return SQRT_SPECIAL_NEG_THIRD

    p = s 
    for _ in range(max(1, int(iterations))):
        p2 = p * p
        denominator = 3.0 * p2 + s
        
        if abs(denominator) < 1e-12:
            break
            
        p = p * (p2 + 3.0 * s) / denominator

    return _to_pair(p)

def pade_cbrt(z: Number, iterations: int = 5) -> ComplexPair:
    s = _as_complex(z)
    if s == 0:
        return (0.0, 0.0)

    if abs(s.real) < 1e-9 and abs(s.imag**2 - 0.5) < 1e-9:
        if s.imag > 0:
            return CBRT_SPECIAL_POS_I
        else:
            return CBRT_SPECIAL_NEG_I

    p = s 
    for _ in range(max(1, int(iterations))):
        p3 = p * p * p
        denominator = 2.0 * p3 + s
        
        if abs(denominator) < 1e-12:
            break
            
        p = p * (p3 + 2.0 * s) / denominator

    return _to_pair(p)

# ------------------------------
# Pade [2/2] direct update formulas
# These implement one-step Pade 2/2 refinements of the form
#   x1 = x0 * (N(s, x0) / D(s, x0))
# where s is the input value and x0 is the current estimate.
# The formulas match the displayed Pade 2/2 approximants where
# S is the original number `s` (not normalized).
# ------------------------------

def pade22_sqrt_step(s_val: Number, x0: Number) -> ComplexPair:
    """One Pade [2/2] update for sqrt(s).

    Formula:
      x1 = x0 * (5*s^2 + 10*s*x0^2 + x0^4) / (s^2 + 10*s*x0^2 + 5*x0^4)

    Returns a pair (real, imag).
    """
    s = _as_complex(s_val)
    x0c = _as_complex(x0)

    # build numerator and denominator
    num = 5.0 * (s * s) + 10.0 * (s * (x0c ** 2)) + (x0c ** 4)
    den = (s * s) + 10.0 * (s * (x0c ** 2)) + 5.0 * (x0c ** 4)

    if abs(den) < 1e-18:
        # fallback: return current estimate
        return _to_pair(x0c)

    x1 = x0c * (num / den)
    return _to_pair(x1)

def pade22_cbrt_step(s_val: Number, x0: Number) -> ComplexPair:
    """One Pade [2/2] update for cbrt(s).

    Formula:
      x1 = x0 * (14*s^2 + 35*s*x0^3 + 5*x0^6) / (5*s^2 + 35*s*x0^3 + 14*x0^6)

    Returns a pair (real, imag).
    """
    s = _as_complex(s_val)
    x0c = _as_complex(x0)

    num = 14.0 * (s * s) + 35.0 * (s * (x0c ** 3)) + 5.0 * (x0c ** 6)
    den = 5.0 * (s * s) + 35.0 * (s * (x0c ** 3)) + 14.0 * (x0c ** 6)

    if abs(den) < 1e-18:
        return _to_pair(x0c)

    x1 = x0c * (num / den)
    return _to_pair(x1)

def pade22_sqrt(s_val: Number, x0: Number = 1.0, iterations: int = 1) -> ComplexPair:
    """Apply Pade [2/2] sqrt update `iterations` times starting from `x0`.
    If `x0` is not provided, uses 1.0 as initial guess.
    """
    x = _as_complex(x0)
    for _ in range(max(1, int(iterations))):
        x = complex(*pade22_sqrt_step(s_val, x))
    return _to_pair(x)

def pade22_cbrt(s_val: Number, x0: Number = 1.0, iterations: int = 1) -> ComplexPair:
    """Apply Pade [2/2] cbrt update `iterations` times starting from `x0`.
    If `x0` is not provided, uses 1.0 as initial guess.
    """
    x = _as_complex(x0)
    for _ in range(max(1, int(iterations))):
        x = complex(*pade22_cbrt_step(s_val, x))
    return _to_pair(x)


In [16]:
import random
import cmath
from typing import Tuple, Union

# ==========================================
# KHU VỰC CHẠY TEST (TESTBENCH)
# ==========================================
def run_benchmark(num_tests=10000, iterations=10):
    random.seed(7)
    
    # 500 test số thực (dương, phạm vi rộng từ 0 -> 1000)
    real_samples = [random.uniform(0.0, 1000.0) for _ in range(num_tests // 2)]
    
    # 500 test số phức (cả thực và ảo từ -500 -> 500)
    complex_samples = [complex(random.uniform(-500.0, 500.0), random.uniform(-500.0, 500.0)) for _ in range(num_tests // 2)]
    
    test_cases = real_samples + complex_samples
    
    sqrt_errors = []
    cbrt_errors = []
    threshold = 1e-3 # Ngưỡng chấp nhận sai số cho 5 vòng lặp
    
    sqrt_pass = 0
    cbrt_pass = 0
    
    for z in test_cases:
        s = complex(z) if not isinstance(z, complex) else z
        
        # --- TEST SQRT ---
        my_sqrt = complex(*pade_sqrt(s, iterations=iterations))
        # Đo sai số bằng cách lấy kết quả bình phương trừ đi số gốc (tránh vấn đề đa nghiệm)
        err_sqrt = abs(my_sqrt**2 - s) / (abs(s) if s != 0 else 1) # Sai số tương đối
        sqrt_errors.append(err_sqrt)
        if err_sqrt < threshold:
            sqrt_pass += 1
            
        # --- TEST CBRT ---
        my_cbrt = complex(*pade_cbrt(s, iterations=iterations))
        # Đo sai số bằng cách lấy kết quả lập phương trừ đi số gốc
        err_cbrt = abs(my_cbrt**3 - s) / (abs(s) if s != 0 else 1) # Sai số tương đối
        cbrt_errors.append(err_cbrt)
        if err_cbrt < threshold:
            cbrt_pass += 1

    print(f"Tổng số case đã test: {len(test_cases)} ({num_tests // 2} thực dương, {num_tests // 2} phức)")
    print(f"\n--- KẾT QUẢ CĂN BẬC 2 ({iterations} lần lặp) ---")
    print(f"Số test đạt (sai số tương đối < {threshold}): {sqrt_pass}/{len(test_cases)}")
    print(f"Sai số trung bình: {sum(sqrt_errors)/len(sqrt_errors):.6f}")
    print(f"Sai số lớn nhất : {max(sqrt_errors):.6f}")

    print(f"\n--- KẾT QUẢ CĂN BẬC 3 ({iterations} lần lặp) ---")
    print(f"Số test đạt (sai số tương đối < {threshold}): {cbrt_pass}/{len(test_cases)}")
    print(f"Sai số trung bình: {sum(cbrt_errors)/len(cbrt_errors):.6f}")
    print(f"Sai số lớn nhất : {max(cbrt_errors):.6f}")

# Chạy thử
run_benchmark(10000, 16)

Tổng số case đã test: 10000 (5000 thực dương, 5000 phức)

--- KẾT QUẢ CĂN BẬC 2 (16 lần lặp) ---
Số test đạt (sai số tương đối < 0.001): 10000/10000
Sai số trung bình: 0.000000
Sai số lớn nhất : 0.000000

--- KẾT QUẢ CĂN BẬC 3 (16 lần lặp) ---
Số test đạt (sai số tương đối < 0.001): 10000/10000
Sai số trung bình: 0.000000
Sai số lớn nhất : 0.000000


In [17]:
EPS = complex(-0.5, 0.8660254037844386)

def solve_cubic(a, b, c, d, iterations=5):
    b_n = b / (-3.0 * a)
    c_n = c / (-3.0 * a)
    d_n = d / (-3.0 * a)
    
    delta_0 = b_n**2 + c_n 
    delta_1 = 2.0 * b_n**3 + 3.0 * b_n * c_n + 3.0 * d_n
    delta = delta_1**2 - 4.0 * delta_0**3
    
    roots = []
    
    if delta.real < 0:
        sqrt_mag = pade_sqrt(abs(delta), iterations)
        sqrt_delta = complex(0.0, sqrt_mag[0])
        C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations))
        C_conj = C_val.conjugate()
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val + (EPS**(2 * k)) * C_conj)
    elif abs(4.0 * delta_0**3) < 1e-9 and delta_1 <= 0:
        C_val = complex(*pade_cbrt(delta_1, iterations))
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val)
    else:
        sqrt_delta = complex(*pade_sqrt(delta, iterations))
        C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations))
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val + (EPS**(2 * k)) * (delta_0 / C_val))
            
    return roots

In [18]:
import random
import numpy as np

def sort_roots(roots):
    return sorted([complex(r) for r in roots], key=lambda x: (round(x.real, 4), round(x.imag, 4)))

def _fmt_c(z):
    z = complex(z)
    return f"({z.real:.6f}{'+' if z.imag>=0 else '-'}{abs(z.imag):.6f}j)"


def diagnose_cubic(a, b, c, d, iterations=10):
    """Compute diagnostic values used by solve_cubic for given coefficients.
    Returns a dict with delta_0, delta_1, delta, sqrt_delta (pair), C_val (complex or None).
    """
    b_n = b / (-3.0 * a)
    c_n = c / (-3.0 * a)
    d_n = d / (-3.0 * a)

    delta_0 = b_n**2 + c_n
    delta_1 = 2.0 * b_n**3 + 3.0 * b_n * c_n + 3.0 * d_n
    delta = delta_1**2 - 4.0 * delta_0**3

    # Try to compute sqrt and cbrt using the Pade 2/2 helpers if available, else fall back
    try:
        sqrt_pair = pade22_sqrt(delta, iterations=iterations)
        sqrt_delta = complex(*sqrt_pair)
    except Exception:
        sqrt_pair = pade_sqrt(delta, iterations)
        sqrt_delta = complex(*sqrt_pair)

    C_val = None
    try:
        C_pair = pade22_cbrt((delta_1 + sqrt_delta) / 2.0, iterations=iterations)
        C_val = complex(*C_pair)
    except Exception:
        try:
            C_pair = pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations)
            C_val = complex(*C_pair)
        except Exception:
            C_val = None

    return {
        'b_n': b_n,
        'c_n': c_n,
        'd_n': d_n,
        'delta_0': delta_0,
        'delta_1': delta_1,
        'delta': delta,
        'sqrt_delta': sqrt_delta,
        'C_val': C_val,
    }


def run_cubic_benchmark(num_tests=1000, iterations=10, report_failures=False, max_report=20):
    random.seed(7)
    pass_count = 0
    threshold = 1e-2
    max_err = 0.0

    failures = []

    for i in range(1, num_tests+1):
        a = random.uniform(-10.0, 10.0)
        while abs(a) < 1e-3:
            a = random.uniform(-10.0, 10.0)
        b = random.uniform(-10.0, 10.0)
        c = random.uniform(-10.0, 10.0)
        d = random.uniform(-10.0, 10.0)

        my_roots = solve_cubic(a, b, c, d, iterations=iterations)
        std_roots = np.roots([a, b, c, d])

        my_roots_sorted = sort_roots(my_roots)
        std_roots_sorted = sort_roots(std_roots)

        case_err = 0.0
        for mr, sr in zip(my_roots_sorted, std_roots_sorted):
            err = abs(mr - sr)
            if err > case_err:
                case_err = err
        
        if case_err > max_err:
            max_err = case_err

        if case_err < threshold:
            pass_count += 1
        else:
            # collect failing case
            if report_failures:
                diag = diagnose_cubic(a, b, c, d, iterations=iterations)
                failures.append({
                    'index': i,
                    'a': a, 'b': b, 'c': c, 'd': d,
                    'err': case_err,
                    'my_roots': my_roots_sorted,
                    'std_roots': std_roots_sorted,
                    'diag': diag,
                })

    print(f"Tổng số test: {num_tests}")
    print(f"Số test khớp kết quả (sai số < {threshold}): {pass_count}/{num_tests}")
    print(f"Sai số lớn nhất ghi nhận: {max_err:.6f}")

    if report_failures and failures:
        print("\n--- Danh sách các test không đạt (giới hạn: {}/{} ) ---".format(min(max_report, len(failures)), len(failures)))
        # print summary table header
        header = f"{'#':>4}  {'a':>9}  {'b':>9}  {'c':>9}  {'d':>9}  {'max_err':>10}"
        print(header)
        print('-' * len(header))
        for f in failures[:max_report]:
            print(f"{f['index']:4d}  {f['a']:9.4f}  {f['b']:9.4f}  {f['c']:9.4f}  {f['d']:9.4f}  {f['err']:10.6e}")

        # print detailed root comparisons and diagnostics for first few failures
        print('\n--- Chi tiết các nghiệm (my_vs_np.roots) và chẩn đoán cho các test đầu ---')
        for f in failures[:min(5, max_report)]:
            d = f['diag']
            print(f"\nTest #{f['index']}: a={f['a']:.6f}, b={f['b']:.6f}, c={f['c']:.6f}, d={f['d']:.6f}, err={f['err']:.6e}")
            print(f"  my_roots : {[ _fmt_c(r) for r in f['my_roots'] ]}")
            print(f"  np.roots : {[ _fmt_c(r) for r in f['std_roots'] ]}")
            print(f"  delta_0  : {d['delta_0']}")
            print(f"  delta_1  : {d['delta_1']}")
            print(f"  delta    : {d['delta']}")
            try:
                print(f"  sqrt_delta: {d['sqrt_delta']}")
            except Exception:
                pass
            try:
                print(f"  C_val    : {d['C_val']}")
            except Exception:
                pass

# Example quick run (smaller number to inspect failures):
run_cubic_benchmark(100000, iterations=25, report_failures=True, max_report=10)


Tổng số test: 100000
Số test khớp kết quả (sai số < 0.01): 100000/100000
Sai số lớn nhất ghi nhận: 0.001810
